In [ ]:
from time import sleep as delay
from project.el5600 import project
if "ic" in globals():
    ic.close()
ic = project(device="el5600", revision="a1", emulator="cp2112", logging=False, is_gui=False)

from phy.multimeter import siglent_sdm3055_auto
from phy.power_supply import keithley_2460, rigol_dp821a, keysight_e36232a
from phy.scope import tektronix_dpo4104b_usb
from phy.eloader import sdl1020x
from phy.relay_16ch import relay_box
# from phy.chamber_th3 import chamber

%matplotlib tk
from interface.docs.output_chart import plot
from interface.cui_colors import color
from interface.i2c_bridge.tca9548a import tca9548
from interface.docs.output_excel import excel_frame, style
from interface.cui_logger import logger as log

import numpy as np

chart = plot()

dm = siglent_sdm3055_auto()
ps = rigol_dp821a()
ks = keysight_e36232a()
kt = keithley_2460()
ld = sdl1020x()
sc = tektronix_dpo4104b_usb()

# dma = agilent_34401a("COM5")
relay = relay_box(i2c_h=ic)
# tc = chamber(port=3)
# mux = tca9548(ic.i2c_h, 0x70)


# ==================================
def disable_all_ps(kt=kt, ps=ps):
    kt.disable
    ks.disable
    ld.disable
    ps.ch1.disable
    ps.ch2.disable
# ==================================

disable_all_ps()

# Test A12

ks : ---> relay.ch1 (VIN)

kt : VDD3V3

ps.ch1 : relay

ps.ch2 : ---> relay.ch3 (EN)

relay.ch2 : terminal_1=VOUT, terminal_2=loader

relay.ch3 : EN

loader : <--- relay.ch2

In [ ]:
temperature = "N40C"

log.output_set_filename(f"[{temperature}_12] I_LIM_RANGE")
log.output_csv(["VIN (V)", "VDD3V3 (V)", "EN (V)", "I_LOAD (A)", "SW_STS", "OFF_BY_ILIM", "ILIM_HYS", "ILIM_TH", "ILIM_TH_EX", "ACTIVE_ILIM_EN"])

In [ ]:
disable_all_ps()
relay.init_channels


v_vdd = 3.3
v_en = 3.3

# --------------------------------------------
ps.ch1.cfg_all = 5, 1
ps.ch1.enable

# --------------------------------------------
kt.cfg_all = v_vdd, 0.01 # vdd
kt.enable
delay(0.5)

# --------------------------------------------
ps.ch2.cfg_all = v_en, 0.1
ps.ch2.enable
delay(0.5)
relay.ch3.enable # en
delay(0.5)
# --------------------------------------------
ks.cfg_all = 12, 6.5
ks.power_recycle
relay.enable = 1, 2

# --------------------------------------------
ic.ILIM_HYS = 0 # 0mA
ic.ILIM_TH = 0
ic.ILIM_TH_EX = 0
ic.ACTIVE_ILIM_EN = 0 # latch

In [ ]:
vin = [12, 20, 28]

# current : [ILIM_TH, ILIM_TH_EX]
dict_load = {
    3.3 : [2, 0],
    4.1 : [2, 1],
    5   : [27, 0],
    5.5 : [15, 1],
    5.9 : [31, 1]
}

for voltage in vin:
    
    ks.cfg_all = voltage, 6.5
    
    for current, reg in dict_load.items():
        
        i_start  = current * 0.9
        i_finish = current + 0.5
        step = 0.01
        ndigit = 2

        list_temp = list(np.arange(i_start, i_finish, step))
        list_load = [round(num, ndigit) for num in list_temp]

        print(f"current step : {step}, round : {ndigit}")
        
        ic.ILIM_TH = reg[0]
        ic.ILIM_TH_EX = reg[1]
        
        ld.iset = list_load[0] * 0.9
        ld.enable
        delay(0.5)
        
        for set_load in list_load:
            
            ld.iset = set_load
            
            try:
                off_by_lim = ic.OFF_BY_ILIM
                sw_sts = ic.SW_STS
                ilim_hys = ic.ILIM_HYS
                ilim_th = ic.ILIM_TH
                ilim_th_ex = ic.ILIM_TH_EX
                active_ilim_en = ic.ACTIVE_ILIM_EN
            except:
                off_by_lim = "NACK"
                sw_sts = "NACK"
                ilim_hys = "NACK"
                ilim_th = "NACK"
                ilim_th_ex = "NACK"
                active_ilim_en = "NACK"
            
            iout = ld.current
            
            ret = [voltage, v_vdd, v_en, iout, sw_sts, off_by_lim, ilim_hys, ilim_th, ilim_th_ex, active_ilim_en]
            log.output_csv(ret)
            
            print(f"{voltage}, {v_vdd}, {v_en}, {iout}, {sw_sts}, {off_by_lim},{ ilim_hys}, {ilim_th}, {ilim_th_ex}, {active_ilim_en}")
            
            if sw_sts != 2:
                ld.disable
                ps.ch2.power_recycle
                break

# ----------------------------------------------------------------------------

relay.init_channels
disable_all_ps()

# ----------------------------------------------------------------------------
print("██████  ██ ███████  ██████  ██████  ███    ██ ███    ██ ███████  ██████ ████████     ██       ██████   █████  ██████  ")
print("██   ██ ██ ██      ██      ██    ██ ████   ██ ████   ██ ██      ██         ██        ██      ██    ██ ██   ██ ██   ██ ")
print("██   ██ ██ ███████ ██      ██    ██ ██ ██  ██ ██ ██  ██ █████   ██         ██        ██      ██    ██ ███████ ██   ██ ")
print("██   ██ ██      ██ ██      ██    ██ ██  ██ ██ ██  ██ ██ ██      ██         ██        ██      ██    ██ ██   ██ ██   ██ ")
print("██████  ██ ███████  ██████  ██████  ██   ████ ██   ████ ███████  ██████    ██        ███████  ██████  ██   ██ ██████  ")
# ----------------------------------------------------------------------------